In [7]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/nikunjnawal009/primevul-graph/primevul_test_paired.jsonl
/kaggle/input/datasets/nikunjnawal009/primevul-graph/primevul_train_paired.jsonl
/kaggle/input/datasets/nikunjnawal009/primevul-graph/primevul_valid_paired.jsonl


In [8]:
import os

for root, dirs, files in os.walk('/kaggle/input'):
    for file in files:
        print(os.path.join(root, file))

/kaggle/input/datasets/nikunjnawal009/primevul-graph/primevul_test_paired.jsonl
/kaggle/input/datasets/nikunjnawal009/primevul-graph/primevul_train_paired.jsonl
/kaggle/input/datasets/nikunjnawal009/primevul-graph/primevul_valid_paired.jsonl


In [9]:
!pip install -U transformers datasets accelerate -q

In [5]:
# ================================
# 🚀 PrimeVul Classifier — Auto‑detect dataset (FIXED)
# ================================

import os, json, random, re
import numpy as np
import torch
import torch.nn as nn
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    set_seed,
    EarlyStoppingCallback
)
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report
)
from sklearn.model_selection import train_test_split

# -------------------------------
# 0. Configuration
# -------------------------------
SEED = 42
MODEL_NAME = "microsoft/codebert-base"
MAX_LENGTH = 512
BATCH_SIZE = 8
GRAD_ACC = 2
EPOCHS = 5
LR = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
EARLY_STOPPING_PATIENCE = 2

set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# -------------------------------
# 1. Auto‑find all JSONL files
# -------------------------------
jsonl_files = []
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if f.endswith(".jsonl"):
            jsonl_files.append(os.path.join(root, f))
            print(f"Found: {os.path.join(root, f)}")

if not jsonl_files:
    raise FileNotFoundError("❌ No JSONL files found in /kaggle/input")

# Load all found files into one list
all_data = []
for path in jsonl_files:
    with open(path, "r") as f:
        for line in f:
            all_data.append(json.loads(line))

print(f"Total samples loaded: {len(all_data)}")

# -------------------------------
# 2. Convert to DataFrame & clean
# -------------------------------
df = pd.DataFrame(all_data)

# ---- Column mapping ----
if "code" not in df.columns:
    for col in ["function", "func", "code_snippet"]:
        if col in df.columns:
            df.rename(columns={col: "code"}, inplace=True)
            break
if "label" not in df.columns:
    for col in ["target", "vul", "vulnerable"]:
        if col in df.columns:
            df.rename(columns={col: "label"}, inplace=True)
            break

# Keep only necessary columns
df = df[["code", "label"]].dropna()

# ---- Label conversion ----
if df["label"].dtype == object:
    df["label"] = df["label"].apply(
        lambda x: int(x) if str(x).isdigit() else (1 if str(x).lower() == "true" else 0)
    )
else:
    df["label"] = df["label"].astype(int)
df["label"] = df["label"].clip(0, 1)   # force binary

# ---- Code cleaning ----
def clean_code(code_str):
    code_str = re.sub(r'//.*', '', code_str)
    code_str = re.sub(r'/\*.*?\*/', '', code_str, flags=re.DOTALL)
    code_str = re.sub(r'[ \t]+', ' ', code_str)
    code_str = re.sub(r'\n\s*\n', '\n', code_str)
    return code_str.strip()

df["code"] = df["code"].apply(clean_code)

print("Label distribution:")
print(df["label"].value_counts())

# -------------------------------
# 3. Stratified train / test split
# -------------------------------
train_df, test_df = train_test_split(
    df, test_size=0.15, stratify=df["label"], random_state=SEED
)
print(f"Train: {len(train_df)}  Test: {len(test_df)}")

# -------------------------------
# 4. Tokenization
# -------------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(examples):
    return tokenizer(
        examples["code"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH
    )

train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
test_ds  = Dataset.from_pandas(test_df.reset_index(drop=True))

train_tokenized = train_ds.map(tokenize_fn, batched=True)
test_tokenized  = test_ds.map(tokenize_fn, batched=True)

train_tokenized.set_format("torch", columns=["input_ids", "attention_mask", "label"])
test_tokenized.set_format("torch", columns=["input_ids", "attention_mask", "label"])

# -------------------------------
# 5. Tiny overfitting test
# -------------------------------
print("\n🧪 Tiny overfitting test (20 samples)...")
tiny_train = train_tokenized.select(range(min(20, len(train_tokenized))))
tiny_test  = test_tokenized.select(range(min(10, len(test_tokenized))))

tiny_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
tiny_args = TrainingArguments(
    output_dir="./tiny_test",
    num_train_epochs=30,
    per_device_train_batch_size=4,
    learning_rate=2e-5,
    logging_steps=1,
    eval_strategy="no",
    save_strategy="no",
    disable_tqdm=True,
    report_to="none",
    seed=SEED
)
tiny_trainer = Trainer(
    model=tiny_model,
    args=tiny_args,
    train_dataset=tiny_train,
    eval_dataset=tiny_test,
)
tiny_trainer.train()
tiny_preds = tiny_trainer.predict(tiny_train)
tiny_loss = tiny_preds.metrics["test_loss"]
print(f"Tiny loss: {tiny_loss:.4f} (should be < 0.1)")

if tiny_loss > 0.5:
    print("⚠️ Loss is high – possible data quality issues. Continuing, but inspect results.")
else:
    print("✅ Overfitting test passed.")

del tiny_model, tiny_trainer, tiny_args
torch.cuda.empty_cache()

# -------------------------------
# 6. Class weights
# -------------------------------
class_counts = train_df["label"].value_counts().sort_index().values
total = class_counts.sum()
class_weights = torch.tensor(
    [total / (2.0 * c) if c > 0 else 1.0 for c in class_counts],
    dtype=torch.float32
).to(device)
print(f"Class weights: {class_weights}")

# -------------------------------
# 7. Custom Trainer with weighted CE
# -------------------------------
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss = nn.functional.cross_entropy(logits, labels, weight=self.class_weights)
        return (loss, outputs) if return_outputs else loss

# -------------------------------
# 8. Full training
# -------------------------------
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model.to(device)

training_args = TrainingArguments(
    output_dir="./best_model",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    gradient_accumulation_steps=GRAD_ACC,
    learning_rate=LR,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    fp16=torch.cuda.is_available(),
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=1,
    report_to="none",
    seed=SEED,
    dataloader_drop_last=False,
    lr_scheduler_type="cosine",
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='binary', zero_division=0
    )
    acc = accuracy_score(labels, preds)
    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
)
trainer.class_weights = class_weights

print("\n🚀 Training with class‑weighted CE...")
trainer.train()

# -------------------------------
# 9. Evaluation
# -------------------------------
print("\n📊 Final test performance:")
test_results = trainer.evaluate(test_tokenized)
for k, v in test_results.items():
    print(f"  {k}: {v:.4f}")

preds_output = trainer.predict(test_tokenized)
y_true = preds_output.label_ids
y_pred = np.argmax(preds_output.predictions, axis=1)

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=["Safe (0)", "Vulnerable (1)"]))

# -------------------------------
# 10. Save model
# -------------------------------
save_path = "/kaggle/working/primevul_codebert"
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)
print(f"\n✅ Model saved to {save_path}")

# -------------------------------
# 11. Quick sanity check
# -------------------------------
sample_safe = "int add(int a, int b) { return a + b; }"
sample_vuln = "int copy(char *src) { char buf[10]; strcpy(buf, src); return 0; }"

for code, lbl in [(sample_safe, "safe"), (sample_vuln, "vulnerable")]:
    inputs = tokenizer(code, return_tensors="pt", truncation=True, max_length=MAX_LENGTH)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    model.eval()
    with torch.no_grad():
        logits = model(**inputs).logits
        prob = torch.softmax(logits, dim=1)[0].cpu().tolist()
        pred = torch.argmax(logits, dim=1).item()
    print(f"\n🔮 {lbl.upper()} example: predicted {'VULNERABLE' if pred==1 else 'SAFE'} "
          f"(confidence: {max(prob):.2%})")

Using device: cuda
Found: /kaggle/input/datasets/nikunjnawal009/primevul-graph/primevul_test_paired.jsonl
Found: /kaggle/input/datasets/nikunjnawal009/primevul-graph/primevul_train_paired.jsonl
Found: /kaggle/input/datasets/nikunjnawal009/primevul-graph/primevul_valid_paired.jsonl
Total samples loaded: 9408
Label distribution:
label
1    4704
0    4704
Name: count, dtype: int64
Train: 7996  Test: 1412


Map:   0%|          | 0/7996 [00:00<?, ? examples/s]

Map:   0%|          | 0/1412 [00:00<?, ? examples/s]


🧪 Tiny overfitting test (20 samples)...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


📊 Final test performance:


  eval_loss: 0.6930
  eval_accuracy: 0.4936
  eval_f1: 0.6483
  eval_precision: 0.4966
  eval_recall: 0.9334
  eval_runtime: 25.0289
  eval_samples_per_second: 56.4150
  eval_steps_per_second: 1.7980
  epoch: 5.0000

Confusion Matrix:
[[ 38 668]
 [ 47 659]]

Classification Report:
                precision    recall  f1-score   support

      Safe (0)       0.45      0.05      0.10       706
Vulnerable (1)       0.50      0.93      0.65       706

      accuracy                           0.49      1412
     macro avg       0.47      0.49      0.37      1412
  weighted avg       0.47      0.49      0.37      1412



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✅ Model saved to /kaggle/working/primevul_codebert

🔮 SAFE example: predicted VULNERABLE (confidence: 54.61%)

🔮 VULNERABLE example: predicted VULNERABLE (confidence: 54.54%)
